In [4]:
import json
from datetime import date, timedelta

import geopandas as gpd
import h3
import polars as pl
import polars_h3 as plh3
import yaml


def update(data: pl.DataFrame, dissolved: list[str]) -> pl.DataFrame:
    return data.with_columns(
        source=pl.when(pl.col("source").is_in(dissolved))
        .then(pl.col("source"))
        .otherwise(plh3.cell_to_parent("source", 7)),
        target=pl.when(pl.col("target").is_in(dissolved))
        .then(pl.col("target"))
        .otherwise(plh3.cell_to_parent("target", 7)),
    )


def to_locations(hexagons: list[str]) -> pl.DataFrame:
    records = []
    for h in hexagons:
        lat, lon = h3.cell_to_latlng(h)
        records.append([h, lat, lon])
    return pl.from_records(records, schema=["id", "lat", "lon"], orient="row")

In [5]:
with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [6]:
RES = 8
DIR = f"{config['dirs']['output']}/{config['city']}"
AGGREGATION = "device"  # device | trip

In [ ]:
# mixed_grid = gpd.read_file("../output/grid/multi_grid_company_distribution.geojson")
# ids = pl.from_pandas(mixed_grid[["h3"]])

# to_locations(ids["h3"].to_list()).write_csv("../output/network_mixedgrid/locations.csv")

# cartesian_product = ids.join(ids, how="cross")
# cartesian_product.columns = ["source", "target"]
# cartesian_product = cartesian_product.with_columns(weight=pl.lit(0))


# budapest_h3_8 = gpd.read_file("../data/budapest_h3_8.geojson")
# in_budapest = budapest_h3_8["h3_polyfill"].tolist()

# with open("../output/grid/dissolved.json", "r") as fp:
#     dissolved = json.load(fp)["dissolved"]

In [7]:
st_at_hw = pl.read_csv(
    f"{DIR}/dynamic_commuting/{RES}/number_of_stops_at_home_work.csv"
)

In [9]:
st_at_hw.head(2)

device_id,day,stops_at_home,stops_at_work,h3_8_home,count_home,h3_8_work,count_work
i64,str,i64,i64,str,i64,str,i64
1,"""2025-05-04""",1,0,"""881e037857fffff""",53,"""881e037857fffff""",35
1,"""2025-05-05""",1,2,"""881e037857fffff""",53,"""881e037857fffff""",35


In [ ]:
# st_at_hw.rename({f"h3_{RES}_home": "h3-home", f"h3_{RES}_work": "h3_work"})

In [13]:
if AGGREGATION == "trip":
    dyn_comm = (
        st_at_hw.filter(
            (pl.col.count_home >= 5)
            & (pl.col.count_work >= 5)
            & (pl.col.stops_at_work > 0)
            # & (pl.col.h3_home.is_in(in_budapest))
            # & (pl.col.h3_work.is_in(in_budapest))
        )
        .group_by(["day", f"h3_{RES}_home", f"h3_{RES}_work"])
        .agg(pl.col.device_id.count().alias("count"))
    )
elif AGGREGATION == "device":
    dyn_comm = (
        st_at_hw.filter(
            (pl.col.count_home >= 5)
            & (pl.col.count_work >= 5)
            & (pl.col.stops_at_work > 0)
            # & (pl.col.h3_home.is_in(in_budapest))
            # & (pl.col.h3_work.is_in(in_budapest))
        )
        .group_by(["day", f"h3_{RES}_home", f"h3_{RES}_work"])
        .agg(pl.col.device_id.unique().count().alias("count"))
    )

In [17]:
dyn_comm.sort(["day", f"h3_{RES}_home", f"h3_{RES}_work"]).write_csv(
    f"{DIR}/dynamic_commuting/{RES}/daily_commuting.csv"
)

In [14]:
dyn_comm.to_pandas()["count"].describe()

count    2.358566e+06
mean     3.029448e+00
std      5.113241e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      3.000000e+00
max      1.130000e+02
Name: count, dtype: float64